# PyKiwoom-REST v2.2.0 종합 테스트 노트북

이 노트북은 PyKiwoom-REST 라이브러리의 주요 기능들을 테스트하고 시연합니다.

## 📋 목차
1. 환경 설정 및 초기화
2. 인증 API 테스트
3. 주식 시세 API 테스트
4. 차트 데이터 API 테스트 (신규 ka10005 포함)
5. 투자자 매매 동향 API 테스트 (수정된 TR 코드)
6. WebSocket 실시간 시세 테스트 (v2.2.0 신규 기능)
7. 계좌 정보 조회
8. 순위 정보 조회
9. 종합 분석 예제

## 1. 환경 설정 및 초기화

먼저 필요한 라이브러리를 임포트하고 API를 초기화합니다.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import time

# PyKiwoom-REST 임포트
from pykiwoom_rest import KiwoomRest

# .env 파일 로드
env_path = (Path.cwd().parent / ".env").resolve()
load_dotenv(dotenv_path=env_path)

# 환경변수 로드
ACCOUNT_NO = os.getenv("ACCOUNT_NO")
KIWOOM_APPKEY = os.getenv("KIWOOM_APPKEY")
KIWOOM_APPSECRET = os.getenv("KIWOOM_APPSECRET")

print("✅ 환경변수 로드 완료")
print(f"   - ACCOUNT_NO: {ACCOUNT_NO[:4]}****" if ACCOUNT_NO else "   - ACCOUNT_NO: 미설정")
print(f"   - APPKEY: {KIWOOM_APPKEY[:10]}..." if KIWOOM_APPKEY else "   - APPKEY: 미설정")

✅ 환경변수 로드 완료
   - ACCOUNT_NO: 6351****
   - APPKEY: Ayr_sSVO7m...


In [2]:
# KiwoomRest 초기화
kiwoom = KiwoomRest(
    account_no=ACCOUNT_NO,
    appkey=KIWOOM_APPKEY,
    appsecret=KIWOOM_APPSECRET,
    env_path=env_path,
    use_mock=False,
)

if kiwoom:
    print("✅ KiwoomRest API 초기화 완료")
    print(f"   - Base URL: {kiwoom.stock_api.api_base.base_url}")


✅ KiwoomRest API 초기화 완료
   - Base URL: https://api.kiwoom.com


## 2. 인증 API 테스트

OAuth2 토큰 발급 및 관리를 테스트합니다.

In [3]:
# 토큰 발급 및 상태 확인
# 먼저 명시적으로 토큰을 발급받습니다
print("🔑 토큰 발급 중...")
try:
    token_info = kiwoom.get_access_token()
    print(f"✅ 토큰 발급 완료")
    print(f"   - 토큰 타입: {token_info.get('token_type', 'N/A')}")
    print(f"   - 유효기간: {token_info.get('expires_in', 0)}초 (24시간)")
    print(f"   - 발급 시각: {token_info.get('issued_at', 'N/A')}")
except Exception as e:
    print(f"❌ 토큰 발급 실패: {e}")

# 토큰 상태 확인 (모든 API 인스턴스에서 토큰 집계)
print("\n📋 현재 토큰 상태:")
token_status = kiwoom.get_token_status()

print(f"   - 토큰 보유: {token_status['has_token']}")
print(f"   - 유효: {token_status['is_valid']}")
print(f"   - 토큰 프리픽스: {token_status['token_prefix']}")
print(f"   - 만료 시간: {token_status['expires_at']}")
print(f"   - 남은 시간: {token_status['time_to_expiry']}초 ({token_status['time_to_expiry'] // 3600}시간 {(token_status['time_to_expiry'] % 3600) // 60}분)")
print(f"   - 갱신 필요: {token_status['needs_refresh']}")

# 토큰 검증
if not token_status['has_token']:
    print("\n⚠️  경고: 토큰이 없습니다.")
elif not token_status['is_valid']:
    print("\n⚠️  경고: 토큰이 만료되었습니다.")
elif token_status['needs_refresh']:
    print("\n🔄 토큰 갱신 권장 (5분 미만 남음)")
else:
    hours = token_status['time_to_expiry'] // 3600
    minutes = (token_status['time_to_expiry'] % 3600) // 60
    print(f"\n✅ 토큰이 정상입니다. {hours}시간 {minutes}분 후 만료됩니다.")


🔑 토큰 발급 중...
✅ 토큰 발급 완료
   - 토큰 타입: Bearer
   - 유효기간: 86400초 (24시간)
   - 발급 시각: 2025-10-29T21:26:49.447804

📋 현재 토큰 상태:
   - 토큰 보유: True
   - 유효: True
   - 토큰 프리픽스: lG3tlb3QNG5MyiP3QngT...
   - 만료 시간: 2025-10-30T21:21:49.447798
   - 남은 시간: 86099초 (23시간 54분)
   - 갱신 필요: False

✅ 토큰이 정상입니다. 23시간 54분 후 만료됩니다.


## 3. 주식 시세 API 테스트

기본 시세 정보를 조회합니다.

In [4]:
# Fail-fast: 필수 필드가 없으면 KeyError 발생
# API 응답이 예상과 다를 때 즉시 문제를 파악할 수 있습니다.
# 삼성전자 시세 조회
stock_code = "005930"
stock_name = "삼성전자"

print(f"📊 {stock_name}({stock_code}) 현재가 조회")
print("=" * 60)

price_info = kiwoom.get_stock_price(stock_code)

if price_info:
    # ka10001 응답은 output 키 없이 바로 데이터가 반환됨
    current_price = int(price_info['cur_prc'].replace('+', '').replace('-', ''))
    change_price = int(price_info['pred_pre'].replace('+', '').replace('-', ''))
    change_rate = float(price_info['flu_rt'].replace('+', '').replace('-', ''))
    volume = int(price_info['trde_qty'].replace(',', ''))
    
    # 등락 부호 판단
    pre_sig = price_info.get('pre_sig', '3')
    if pre_sig == '2':
        sign = '+'
    elif pre_sig == '5':
        sign = '-'
    else:
        sign = ''
    
    print(f"\n💰 현재가: {current_price:,}원")
    print(f"📈 전일대비: {sign}{change_price:,}원 ({sign}{change_rate:.2f}%)")
    print(f"📊 거래량: {volume:,}주")
    print(f"🕐 조회시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # 추가 정보
    print(f"\n📋 추가 정보:")
    print(f"   시가: {int(price_info['open_pric'].replace('+', '').replace('-', '')):,}원")
    print(f"   고가: {int(price_info['high_pric'].replace('+', '').replace('-', '')):,}원")
    print(f"   저가: {int(price_info['low_pric'].replace('+', '').replace('-', '')):,}원")
    print(f"   외국인보유비율: {price_info['for_exh_rt']}%")
else:
    print("❌ 시세 조회 실패")

📊 삼성전자(005930) 현재가 조회

💰 현재가: 100,500원
📈 전일대비: +1,000원 (+1.01%)
📊 거래량: 20,899,788주
🕐 조회시간: 2025-10-29 21:26:49

📋 추가 정보:
   시가: 100,200원
   고가: 101,000원
   저가: 99,100원
   외국인보유비율: +52.26%


In [5]:
# Fail-fast: 필수 필드가 없으면 KeyError 발생
# API 응답이 예상과 다를 때 즉시 문제를 파악할 수 있습니다.
# 호가 정보 조회
print(f"\n📋 {stock_name}({stock_code}) 호가 정보")
print("=" * 60)

orderbook = kiwoom.get_stock_orderbook(stock_code)

if orderbook and 'output' in orderbook:
    output = orderbook['output']
    
    print("\n매도 호가:")
    for i in range(1, 6):
        ask_price = int(output.get(f'askp{i}', 0))
        ask_volume = int(output.get(f'askp_rsqn{i}', 0))
        print(f"  {i}. {ask_price:>8,}원 | {ask_volume:>10,}주")
    
    print("\n매수 호가:")
    for i in range(1, 6):
        bid_price = int(output.get(f'bidp{i}', 0))
        bid_volume = int(output.get(f'bidp_rsqn{i}', 0))
        print(f"  {i}. {bid_price:>8,}원 | {bid_volume:>10,}주")
else:
    print("❌ 호가 조회 실패")


📋 삼성전자(005930) 호가 정보
❌ 호가 조회 실패


## 4. 차트 데이터 API 테스트

### 4.1 새로운 통합 차트 API (ka10005) 테스트

In [6]:
# 새로운 통합 차트 API 테스트 (ka10005 - 이전에 stock_api에서 chart_api로 이동)
print(f"📊 {stock_name} 통합 차트 조회 (ka10005)")
print("=" * 60)

# 일봉 차트
daily_chart = kiwoom.chart_api.get_daily_weekly_monthly_minute_chart(
    stock_code=stock_code,
    period="D",  # D=일봉
    start_date=(datetime.now() - timedelta(days=30)).strftime("%Y%m%d"),
    end_date=datetime.now().strftime("%Y%m%d")
)

print("\n📈 일봉 데이터 (최근 5일):")
if daily_chart and 'stk_ddwkmm' in daily_chart:  # 'output'이 아니라 'stk_ddwkmm'
    df = pd.DataFrame(daily_chart['stk_ddwkmm'][:5])
    print(df[['date', 'open_pric', 'high_pric', 'low_pric', 'close_pric', 'trde_qty']].to_string(index=False))  # 'volume'이 아니라 'trde_qty'
else:
    print("❌ 일봉 데이터 조회 실패")
    print(f"   응답: {daily_chart}")

📊 삼성전자 통합 차트 조회 (ka10005)

📈 일봉 데이터 (최근 5일):
    date open_pric high_pric low_pric close_pric trde_qty
20251029   +100200   +101000   -99100    +100500 20899788
20251028    100900    101000    99100     -99500 20002282
20251027    101300    102000   100600    +102000 22169970
20251024     97900     99000    97700     +98800 18801925
20251023     96800     98500    96300     -96500 18488581


### 4.2 기존 차트 API 테스트

In [14]:
# Fail-fast: 필수 필드가 없으면 KeyError 발생
# API 응답이 예상과 다를 때 즉시 문제를 파악할 수 있습니다.
# 분봉 차트 조회
print(f"\n📊 {stock_name} 5분봉 차트 (최근 20개)")
print("=" * 60)

minute_chart = kiwoom.get_minute_chart(
    stock_code=stock_code,
    interval=5,
    count=20
)

if minute_chart and 'output2' in minute_chart:
    data = minute_chart['output2'][:10]
    
    print(f"\n{'시간':12} {'현재가':>10} {'등락':>8} {'거래량':>12}")
    print("-" * 60)
    
    for item in data:
        time_str = item['cntr_tm'][:12]  # YYYYMMDDHHMM
        if len(time_str) >= 12:
            formatted_time = f"{time_str[4:6]}/{time_str[6:8]} {time_str[8:10]}:{time_str[10:12]}"
        else:
            formatted_time = time_str
        
        cur_price = int(item['cur_prc'].replace('+', '').replace('-', ''))
        pred_pre = int(item['pred_pre'].replace('+', '').replace('-', ''))
        volume = int(item['trde_qty'].replace(',', ''))
        
        # 등락 부호
        pred_pre_sig = item.get('pred_pre_sig', '3')
        sign = '+' if pred_pre_sig == '2' else '-' if pred_pre_sig == '5' else ''
        
        print(f"{formatted_time:12} {cur_price:>10,}원 {sign}{pred_pre:>7,} {volume:>12,}주")
else:
    print("❌ 분봉 차트 조회 실패")
    print(f"   응답: {minute_chart}")


📊 삼성전자 5분봉 차트 (최근 20개)
❌ 분봉 차트 조회 실패
   응답: {'stk_cd': '005930', 'stk_min_pole_chart_qry': [{'cur_prc': '+100500', 'trde_qty': '1095888', 'cntr_tm': '20251029153000', 'open_pric': '+100500', 'high_pric': '+100500', 'low_pric': '+100500', 'acc_trde_qty': '20645411', 'pred_pre': '+1000', 'pred_pre_sig': '2'}, {'cur_prc': '+100700', 'trde_qty': '373696', 'cntr_tm': '20251029151500', 'open_pric': '+100800', 'high_pric': '+100800', 'low_pric': '+100600', 'acc_trde_qty': '19549523', 'pred_pre': '+1200', 'pred_pre_sig': '2'}, {'cur_prc': '+100800', 'trde_qty': '300566', 'cntr_tm': '20251029151000', 'open_pric': '+100850', 'high_pric': '+100900', 'low_pric': '+100700', 'acc_trde_qty': '19175827', 'pred_pre': '+1300', 'pred_pre_sig': '2'}, {'cur_prc': '+100800', 'trde_qty': '665518', 'cntr_tm': '20251029150500', 'open_pric': '+100400', 'high_pric': '+100900', 'low_pric': '+100300', 'acc_trde_qty': '18875261', 'pred_pre': '+1300', 'pred_pre_sig': '2'}, {'cur_prc': '+100400', 'trde_qty': '375266

### 4.3 실제 응답 예제 (ka10005)

ka10005 API의 실제 응답 구조를 살펴봅니다.

In [8]:
# ka10005 API 실제 응답 예제
print("📋 ka10005 (주식일주월시분요청) 실제 응답 구조")
print("=" * 60)

# 실제 API 응답 예제 (삼성전자 일봉)
example_response = {
    "stk_ddwkmm": [
        {
            "date": "20241028",
            "open_pric": "95400",
            "high_pric": "95400",
            "low_pric": "95400",
            "close_pric": "95400",
            "pre": "0",
            "flu_rt": "0.00",
            "trde_qty": "0",
            "trde_prica": "0",
            "cntr_str": "0.00",
            "for_poss": "+26.07",
            "for_wght": "+26.07",
            "for_netprps": "0",
            "orgn_netprps": "",
            "ind_netprps": "",
            "frgn": "",
            "crd_remn_rt": "",
            "prm": ""
        },
        {
            "date": "20241025",
            "open_pric": "95400",
            "high_pric": "95400",
            "low_pric": "95400",
            "close_pric": "95400",
            "pre": "",
            "flu_rt": "",
            "trde_qty": "0",
            "trde_prica": "",
            "cntr_str": "",
            "for_poss": "",
            "for_wght": "",
            "for_netprps": "",
            "orgn_netprps": "",
            "ind_netprps": "",
            "frgn": "",
            "crd_remn_rt": "",
            "prm": ""
        },
        {
            "date": "20241024",
            "open_pric": "94300",
            "high_pric": "95400",
            "low_pric": "94300",
            "close_pric": "+95400",
            "pre": "",
            "flu_rt": "",
            "trde_qty": "70",
            "trde_prica": "",
            "cntr_str": "",
            "for_poss": "",
            "for_wght": "",
            "for_netprps": "",
            "orgn_netprps": "",
            "ind_netprps": "",
            "frgn": "",
            "crd_remn_rt": "",
            "prm": ""
        }
    ],
    "return_code": 0,
    "return_msg": "정상적으로 처리되었습니다"
}

import json
print(json.dumps(example_response, indent=2, ensure_ascii=False))

print("\n📊 필드 설명:")
print("-" * 60)
print("date         : 날짜 (YYYYMMDD)")
print("open_pric    : 시가")
print("high_pric    : 고가")
print("low_pric     : 저가")
print("close_pric   : 종가 (+ 기호는 상승 표시)")
print("trde_qty     : 거래량")
print("for_poss     : 외국인 보유량")
print("for_wght     : 외국인 보유 비중")
print("for_netprps  : 외국인 순매수량")
print("-" * 60)

# DataFrame으로 변환하여 보기
print("\n📈 DataFrame 변환 예제:")
df = pd.DataFrame(example_response['stk_ddwkmm'])
print(df[['date', 'open_pric', 'high_pric', 'low_pric', 'close_pric', 'trde_qty']].to_string(index=False))

📋 ka10005 (주식일주월시분요청) 실제 응답 구조
{
  "stk_ddwkmm": [
    {
      "date": "20241028",
      "open_pric": "95400",
      "high_pric": "95400",
      "low_pric": "95400",
      "close_pric": "95400",
      "pre": "0",
      "flu_rt": "0.00",
      "trde_qty": "0",
      "trde_prica": "0",
      "cntr_str": "0.00",
      "for_poss": "+26.07",
      "for_wght": "+26.07",
      "for_netprps": "0",
      "orgn_netprps": "",
      "ind_netprps": "",
      "frgn": "",
      "crd_remn_rt": "",
      "prm": ""
    },
    {
      "date": "20241025",
      "open_pric": "95400",
      "high_pric": "95400",
      "low_pric": "95400",
      "close_pric": "95400",
      "pre": "",
      "flu_rt": "",
      "trde_qty": "0",
      "trde_prica": "",
      "cntr_str": "",
      "for_poss": "",
      "for_wght": "",
      "for_netprps": "",
      "orgn_netprps": "",
      "ind_netprps": "",
      "frgn": "",
      "crd_remn_rt": "",
      "prm": ""
    },
    {
      "date": "20241024",
      "open_pric": "94

## 5. 투자자 매매 동향 API 테스트

### 수정된 TR 코드 (ka10005 → ka10058) 사용

In [9]:
# 투자자별 매매동향 API 이슈 안내
print(f"⚠️  투자자별 매매동향 API 안내")
print("=" * 60)
print("❌ 현재 get_stock_investor_trading() 메서드는 TR 코드 매핑이 잘못되어 있습니다.")
print()
print("📋 문제점:")
print("   - 현재 ka10058(투자자별일별매매종목요청)로 매핑되어 있음")
print("   - ka10058은 특정 종목이 아니라 투자자별 매매 종목 리스트를 조회하는 API")
print("   - stock_code 파라미터 대신 strt_dt, end_dt, trde_tp, mrkt_tp, invsr_tp, stex_tp 필요")
print()
print("✅ 대안:")
print("   1. 외국인 매매동향: get_foreign_trading(stock_code) - ka10008")
print("   2. 기관별 매매동향: get_stock_member_trading(stock_code) - ka10006")
print("   3. 장중 투자자별 매매: 전체 시장 순위 조회 - ka10063")
print()
print("   → Cell 17에서 외국인 매매동향(ka10008)을 확인하세요.")
print("=" * 60)

⚠️  투자자별 매매동향 API 안내
❌ 현재 get_stock_investor_trading() 메서드는 TR 코드 매핑이 잘못되어 있습니다.

📋 문제점:
   - 현재 ka10058(투자자별일별매매종목요청)로 매핑되어 있음
   - ka10058은 특정 종목이 아니라 투자자별 매매 종목 리스트를 조회하는 API
   - stock_code 파라미터 대신 strt_dt, end_dt, trde_tp, mrkt_tp, invsr_tp, stex_tp 필요

✅ 대안:
   1. 외국인 매매동향: get_foreign_trading(stock_code) - ka10008
   2. 기관별 매매동향: get_stock_member_trading(stock_code) - ka10006
   3. 장중 투자자별 매매: 전체 시장 순위 조회 - ka10063

   → Cell 17에서 외국인 매매동향(ka10008)을 확인하세요.


In [10]:
# 외국인 투자자 매매동향 조회 (ka10008)
print(f"\n🌏 {stock_name} 외국인 매매동향 (ka10008)")
print("=" * 60)

foreign_trading = kiwoom.get_foreign_trading(stock_code)

if foreign_trading and 'stk_frgnr' in foreign_trading:
    df = pd.DataFrame(foreign_trading['stk_frgnr'][:10])
    print("최근 10일 외국인 매매 동향:")
    print(df[['dt', 'wght', 'trde_qty', 'chg_qty']].to_string(index=False))  # chg_qty = 변동수량 (순매수)
    
    print("\n📋 필드 설명:")
    print("   dt: 일자")
    print("   wght: 비중 (%)")
    print("   trde_qty: 거래량")
    print("   chg_qty: 변동수량 (순매수량)")
else:
    print("❌ 외국인 매매동향 조회 실패")
    print(f"   응답: {foreign_trading}")


🌏 삼성전자 외국인 매매동향 (ka10008)
최근 10일 외국인 매매 동향:
      dt   wght trde_qty  chg_qty
20251029 +52.26 20899788 -2389981
20251028 +52.30 20002282 -3465819
20251027 +52.36 22169970  8230623
20251024 +52.22 18801925  5286145
20251023 +52.13 18488581 -1135978
20251022 +52.15 15937611   274581
20251021 +52.15 22803830   813014
20251020 +52.13 17589756  -954628
20251017 +52.15 22730809  3778113
20251016 +52.09 28141060  5270864

📋 필드 설명:
   dt: 일자
   wght: 비중 (%)
   trde_qty: 거래량
   chg_qty: 변동수량 (순매수량)


## 6. WebSocket 실시간 시세 테스트 (v2.2.0 신규 기능)

### 주의: 실시간 데이터는 장 시간(09:00-15:30)에만 수신됩니다.

In [11]:
# WebSocket 실시간 시세 테스트
print("🔌 WebSocket 실시간 시세 테스트")
print("=" * 60)
print("⚠️  이 기능은 장 시간(09:00-15:30)에만 데이터를 수신합니다.\n")

# 수신 데이터 저장용
received_quotes = []

def on_quote(quote):
    """실시간 시세 수신 콜백"""
    received_quotes.append(quote)
    print(f"\n📊 실시간 시세: {quote.stock_code}")
    print(f"   현재가: {quote.current_price:,}원")
    print(f"   등락: {quote.change_price:+,}원 ({quote.change_rate:+.2f}%)")
    print(f"   거래량: {quote.volume:,}")
    print(f"   시간: {quote.timestamp}")

try:
    # WebSocket 활성화
    print("1️⃣ WebSocket 연결 중...")
    if kiwoom.enable_websocket():
        print("✅ WebSocket 연결 성공\n")
        
        # 삼성전자 실시간 시세 구독
        print(f"2️⃣ {stock_name} 실시간 시세 구독 중...")
        if kiwoom.subscribe_realtime_quote(stock_code, on_quote):
            print("✅ 실시간 시세 구독 성공\n")
            
            # 10초간 데이터 수신 대기
            print("3️⃣ 10초간 실시간 데이터 수신 대기 중...")
            print("   (장 시간이 아니면 데이터가 수신되지 않습니다)\n")
            
            import asyncio
            for i in range(10):
                time.sleep(1)
                try:
                    loop = asyncio.get_event_loop()
                    loop.run_until_complete(asyncio.sleep(0))
                except:
                    pass
                print(f"   {i+1}/10초 경과...")
            
            # 결과 요약
            print(f"\n✅ 수신 완료: {len(received_quotes)}건")
            if len(received_quotes) == 0:
                print("⚠️  데이터를 수신하지 못했습니다.")
                print("   - 장 시간이 아니거나")
                print("   - WebSocket 엔드포인트 변경이 필요할 수 있습니다.")
            
            # 구독 해제
            print("\n4️⃣ 구독 해제 및 연결 종료...")
            kiwoom.unsubscribe_realtime_all()
            kiwoom.disable_websocket()
            print("✅ 정리 완료")
        else:
            print("❌ 실시간 시세 구독 실패")
    else:
        print("❌ WebSocket 연결 실패")
        
except Exception as e:
    print(f"\n❌ WebSocket 오류: {e}")
    import traceback
    traceback.print_exc()

🔌 WebSocket 실시간 시세 테스트
⚠️  이 기능은 장 시간(09:00-15:30)에만 데이터를 수신합니다.

1️⃣ WebSocket 연결 중...

❌ WebSocket 오류: This event loop is already running


Traceback (most recent call last):
  File "/tmp/ipykernel_777567/4118291410.py", line 21, in <module>
    if kiwoom.enable_websocket():
       ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/unohee/dev/tools/pykiwoom-rest/src/pykiwoom_rest/kiwoom_rest.py", line 684, in enable_websocket
    self._websocket_enabled = loop.run_until_complete(self.websocket.connect())
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/base_events.py", line 663, in run_until_complete
    self._check_running()
  File "/usr/lib/python3.12/asyncio/base_events.py", line 622, in _check_running
    raise RuntimeError('This event loop is already running')
RuntimeError: This event loop is already running
/tmp/ipykernel_777567/4118291410.py:63: RuntimeWarning: coroutine 'WebSocketAPI.connect' was never awaited
  traceback.print_exc()


## 7. 계좌 정보 조회

보유 종목 및 계좌 잔고를 조회합니다.

In [12]:
# 계좌 잔고 조회
print("💼 계좌 잔고 조회")
print("=" * 60)

try:
    balance = kiwoom.get_account_balance()
    
    if balance and 'output1' in balance:
        holdings = balance['output1']
        print(f"\n보유 종목 수: {len(holdings)}개\n")
        
        if holdings:
            df = pd.DataFrame(holdings)
            print(df[['pdno', 'prdt_name', 'hldg_qty', 'pchs_avg_pric', 'prpr', 'evlu_pfls_rt']].to_string(index=False))
        else:
            print("보유 종목이 없습니다.")
    else:
        print("❌ 계좌 잔고 조회 실패")
        
except Exception as e:
    print(f"❌ 오류: {e}")

💼 계좌 잔고 조회
❌ 오류: 'KiwoomRest' object has no attribute 'get_account_balance'


## 8. 순위 정보 조회

거래대금 상위 종목을 조회합니다.

In [13]:
# 거래대금 상위 종목
print("🏆 거래대금 상위 종목 (KOSPI)")
print("=" * 60)

top_amount = kiwoom.get_top_trading_amount(market="KOSPI", count=10)

if top_amount and 'output' in top_amount:
    df = pd.DataFrame(top_amount['output'])
    print(df[['hts_kor_isnm', 'stck_prpr', 'prdy_vrss_sign', 'prdy_ctrt', 'acml_tr_pbmn']].to_string(index=False))
else:
    print("❌ 거래대금 상위 종목 조회 실패")

🏆 거래대금 상위 종목 (KOSPI)


AttributeError: 'KiwoomRest' object has no attribute 'get_top_trading_amount'

## 9. 종합 분석 예제

여러 API를 조합하여 종합적인 종목 분석을 수행합니다.

In [ ]:
# Fail-fast: 필수 필드가 없으면 KeyError 발생
# API 응답이 예상과 다를 때 즉시 문제를 파악할 수 있습니다.
# 삼성전자 시세 조회
stock_code = "005930"
stock_name = "삼성전자"

print(f"📊 {stock_name}({stock_code}) 현재가 조회")
print("=" * 60)

price_info = kiwoom.get_stock_price(stock_code)

if price_info:
    # ka10001 응답은 output 키 없이 바로 데이터가 반환됨
    current_price = int(price_info['cur_prc'].replace('+', '').replace('-', ''))
    change_price = int(price_info['pred_pre'].replace('+', '').replace('-', ''))
    change_rate = float(price_info['flu_rt'].replace('+', '').replace('-', ''))
    volume = int(price_info['trde_qty'].replace(',', ''))
    
    # 등락 부호 판단
    pre_sig = price_info.get('pre_sig', '3')
    if pre_sig == '2':
        sign = '+'
    elif pre_sig == '5':
        sign = '-'
    else:
        sign = ''
    
    print(f"\n💰 현재가: {current_price:,}원")
    print(f"📈 전일대비: {sign}{change_price:,}원 ({sign}{change_rate:.2f}%)")
    print(f"📊 거래량: {volume:,}주")
    print(f"🕐 조회시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # 추가 정보
    print(f"\n📋 추가 정보:")
    print(f"   시가: {int(price_info['open_pric'].replace('+', '').replace('-', '')):,}원")
    print(f"   고가: {int(price_info['high_pric'].replace('+', '').replace('-', '')):,}원")
    print(f"   저가: {int(price_info['low_pric'].replace('+', '').replace('-', '')):,}원")
    print(f"   외국인보유비율: {price_info['for_exh_rt']}%")
else:
    print("❌ 시세 조회 실패")

In [ ]:
# 여러 종목 분석
stocks_to_analyze = [
    ("005930", "삼성전자"),
    ("000660", "SK하이닉스"),
    ("005380", "현대차"),
]

results = []

for stock_code, stock_name in stocks_to_analyze:
    try:
        analysis = analyze_stock_comprehensive(stock_code, stock_name)
        results.append(analysis)
        time.sleep(0.5)  # Rate limit 방지
    except Exception as e:
        print(f"❌ {stock_name} 분석 실패: {e}\n")

# 결과 요약 테이블
if results:
    print("\n" + "="*100)
    print("📊 종합 분석 결과 요약")
    print("="*100)
    df_summary = pd.DataFrame(results)
    print(df_summary.to_string(index=False))

## 10. 정리 및 로그아웃

사용이 끝나면 토큰을 폐기합니다.

In [ ]:
# 로그아웃 (토큰 폐기)
print("👋 로그아웃 중...")

try:
    kiwoom.logout()
    print("✅ 로그아웃 완료")
    print("   토큰이 폐기되었습니다.")
except Exception as e:
    print(f"⚠️  로그아웃 오류: {e}")
    print("   (토큰은 24시간 후 자동 만료됩니다)")

---

## 📚 참고 사항

### v2.2.0 주요 변경사항

1. **WebSocket 실시간 시세 기능 추가**
   - `enable_websocket()`, `subscribe_realtime_quote()` 등
   - 장 시간에만 데이터 수신 가능

2. **TR 코드 매핑 수정**
   - `ka10005`: stock_api → chart_api로 이동 (주식일주월시분요청)
   - `get_stock_investor_trading()`: ka10005 → ka10058로 변경
   - 새로운 차트 메서드: `get_daily_weekly_monthly_minute_chart()`

3. **신규 TR 코드 추가**
   - `ka10008`: 외국인 투자자 매매동향
   - `ka10058`: 투자자별 일별 매매동향
   - `ka10063`: 장중 투자자별 매매동향

### Rate Limiting
- 기본 제한: 초당 20회
- 429 오류 발생 시 자동 재시도
- 연속 요청 시 적절한 딜레이 권장

### 장 시간
- 정규 장: 09:00 - 15:30
- 시간외 단일가: 08:30 - 09:00, 15:40 - 16:00
- 일부 API는 장 시간에만 정상 작동

### 문의 및 이슈
- GitHub: https://github.com/unohee/pykiwoom-rest
- 문서: README.md, API_DOCUMENTATION.md

---

**생성 일시**: 2025-10-29  
**PyKiwoom-REST 버전**: v2.2.0